# **Stage 2 (benchmark) — Delinquency Rate: Multivariate Time-Series Forecast**
## SARIMAX · ARDL · VAR | US Quarterly
___
**Purpose.** Forecast the US delinquency rate (`us_delinquency_rate`) directly with
**time-series methods**, as an interpretable, audit-defensible **benchmark** for the
machine-learning PD model built by the wider team. This notebook is the
time-series counterpart to that ML work — the ML methods are deliberately **out of
scope here**.

**How this consumes Stage 1 (02a).** The single input is `SARIMA_regressors_US_Q.csv`
— the *output* of `02a_Stage1_TimeSeries.ipynb`. That one file already contains both
(i) the historical delinquency target and (ii) the **forecast-period paths** of every
macro regressor. So 02a's forecasts are used exactly as intended: as the **future
exogenous inputs** that the conditional models (SARIMAX, ARDL) need to project the
target beyond the last observation.

| Step | Description | Output |
|------|-------------|--------|
| 0 | Configuration & data loading | — |
| 1 | Sample diagnostics & EDA (COVID transmission break) | — |
| 2 | Model specifications (orders selected once) | — |
| 3 | Out-of-sample backtest + Diebold-Mariano | LaTeX table |
| 4 | 20-quarter forward forecast (2026–2030) | — |
| 5 | Export & limitations | `TS_delinquency_forecast_US_Q.csv` |

**Models.** Naive random walk (DM benchmark) · **AR(1)** (requested baseline) ·
SARIMA (best univariate) · **SARIMAX** and **ARDL** (macro-conditional — these consume
02a's exogenous forecasts) · **VAR** (joint-system cross-check).

---
### Methodological notes — read before interpreting results

1. **"Multivariate" splits into two families that treat 02a differently.**
   *Single-equation conditional* models (SARIMAX, ARDL) take the other variables as
   given inputs, so they genuinely consume 02a's forecasts. *Joint-system* models
   (VAR, and its VECM/BVAR/FAVAR relatives) forecast **all** variables simultaneously
   from their own dynamics — they generate their own macro paths and do **not** consume
   02a. The VAR here is therefore an independent cross-check, not a consumer of Stage 1;
   forcing 02a's paths into it would require hard-conditional forecasting and change the
   model's character. SARIMAX/ARDL are the primary models that match the intended design.

2. **The conditional intervals understate uncertainty.** SARIMAX/ARDL treat 02a's
   forecasted regressors as *known*. The real forecast also carries the uncertainty of
   those upstream macro forecasts, which is not propagated here. Interval bands are shown
   for the conditional mean only; full uncertainty quantification and coherent IFRS 9
   base/upside/downside construction are a **later stage** and are not attempted here.

3. **COVID is excluded from both fitting and evaluation.** During 2020–21, forbearance
   severed the macro→credit link: delinquency *fell* to ~1.5% while unemployment spiked
   to 11%. A macro-conditional model cannot (and should not) explain this. COVID quarters
   are set to `NaN` (state-space handles missing data natively — the gold-standard
   treatment noted in 02a) and every backtest origin whose target lands in the window is
   skipped.

4. **A one-step backtest does not fully validate the 5-year path.** At `h=1` every lagged
   regressor is already realised, giving a clean, fair test. The 20-quarter forward
   forecast, by contrast, leans on 02a's *forecasted* regressors once the horizon exceeds
   each variable's lag. Read the backtest as validating short-horizon conditional skill,
   not the full lifetime path.

*Scope note:* VECM, Bayesian VAR and FAVAR were considered and left out for parsimony at
this sample size (~130 usable quarters) and for audit interpretability; they can be added
if a joint-system extension is wanted.


## **0: Configuration & Data Loading**
___
All user-facing settings live here — target, exogenous block, COVID window,
horizons and output filename. Data is loaded directly from the project GitHub repo
(the same repo Stage 1 writes to), with a local fallback to `../Data Collection/`.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import logging; logging.getLogger('statsmodels').setLevel(logging.ERROR)
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning
warnings.simplefilter('ignore', ConvergenceWarning)   # expected during rolling re-fits
warnings.simplefilter('ignore', ValueWarning)         # date-index freq notices
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pmdarima import auto_arima
from IPython.display import display

# ── GitHub data source (same repo as Stage 1 / 02a) ───────────────────────────
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/hogandan85/ST-498/main/Data%20Collection'
GITHUB_TOKEN    = None
LOCAL_OUTPUT    = Path.cwd()/ 'Data Collection'

# ── Input: the Stage 1 (02a) output ───────────────────────────────────────────
# Contains the historical target AND the forecast-period paths of every macro
# regressor, so 02a's forecasts serve as the future exogenous inputs below.
INPUT_REGRESSORS = 'SARIMA_regressors_US_Q.csv'

# ── Target & exogenous blocks ─────────────────────────────────────────────────
TARGET = 'us_delinquency_rate'
# Six CCF-significant LAGGED predictors (identical to the Stage 2 regressor set).
# Because each is lagged, at h=1 every value is already realised -> the one-step
# backtest uses no forecasted inputs and is a fair test.
EXOG = ['us_house_price_yoy_L3', 'us_consumer_confidence_L2', 'us_unemployment_L1',
        'us_credit_qoq_growth_L6', 'us_gdp_yoy_growth_L0', 'us_cpi_L6']
# Contemporaneous block for the VAR cross-check. A VAR forecasts all of these
# jointly from their own dynamics -> it does NOT consume 02a (see notes in Cell 0).
VAR_VARS = [TARGET, 'us_unemployment', 'us_gdp_yoy_growth', 'us_cpi', 'us_credit_qoq_growth']

# ── COVID exclusion window (closed interval; both endpoints excluded) ──────────
COVID_START, COVID_END = '2020-03-31', '2021-12-31'

# ── Horizons ──────────────────────────────────────────────────────────────────
M          = 4      # seasonal period (quarterly)
H          = 20     # forward horizon (5 years; IFRS 9 lifetime)
BACKTEST_H = 1      # OOS evaluation horizon (1 => cleanest DM; lagged exog all realised)
BACKTEST_MIN_TRAIN = 60

# ── Colour palette (matches 02a) ──────────────────────────────────────────────
NAVY, BLUE, RED, AMBER, GREEN, GREY = (
    '#1F3864', '#2E75B6', '#C0392B', '#E67E22', '#27AE60', '#7F8C8D')
TEMPLATE = 'plotly_white'
RECESSIONS_US = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]

# ── Output filename ───────────────────────────────────────────────────────────
MODEL_PREFIX = 'TS'
OUT_FORECAST = f'{MODEL_PREFIX}_delinquency_forecast_US_Q.csv'

# ── Load the Stage 1 output (GitHub first, local fallback) ─────────────────────
def load_regressors(filename, token=None):
    url = f'{GITHUB_RAW_BASE}/{filename}'
    try:
        if token:
            import requests, io
            r = requests.get(url, headers={'Authorization': f'token {token}'})
            r.raise_for_status()
            out = pd.read_csv(io.StringIO(r.text), index_col=0, parse_dates=True)
        else:
            out = pd.read_csv(url, index_col=0, parse_dates=True)
        print(f'Loaded {filename} from GitHub')
    except Exception as e:
        local = LOCAL_OUTPUT / filename
        out = pd.read_csv(local, index_col=0, parse_dates=True)
        print(f'GitHub load failed ({type(e).__name__}); loaded local copy: {local}')
    return out.asfreq('QE-DEC')

df = load_regressors(INPUT_REGRESSORS, token=GITHUB_TOKEN)
print(f'  Shape : {df.shape}')
print(f'  Span  : {df.index.min().date()} -> {df.index.max().date()}')
print(f'  Target: {TARGET} — {df[TARGET].notna().sum()} obs | '
      f'min {df[TARGET].min():.2f}% | max {df[TARGET].max():.2f}% | mean {df[TARGET].mean():.2f}%')

Loaded SARIMA_regressors_US_Q.csv from GitHub
  Shape : (164, 42)
  Span  : 1990-03-31 -> 2030-12-31
  Target: us_delinquency_rate — 140 obs | min 1.53% | max 6.77% | mean 3.70%


In [2]:
# ── Split history vs forecast horizon on target availability ──────────────────
def cmask(idx):
    idx = pd.DatetimeIndex(idx)
    return (idx >= COVID_START) & (idx <= COVID_END)

hist_idx = df.index[df[TARGET].notna()]          # target observed
fc_idx   = df.index[df.index > hist_idx.max()]   # 20 forecast quarters (target NaN)

# Exogenous block starts once all lag columns are populated (early-sample lags NaN).
X_all = df.loc[hist_idx, EXOG]
start = X_all.dropna().index.min()

y_full = df.loc[start:hist_idx.max(), TARGET]     # observed target, exog-complete window
X_hist = df.loc[start:hist_idx.max(), EXOG]       # historical exog
X_fc   = df.loc[fc_idx, EXOG].interpolate(limit_direction='both')  # 02a forecast exog

# COVID handling: NaN-in-place for the ARIMA family (state-space handles missing
# natively); a COVID-dropped copy is used only where a method cannot take NaN.
y_nan  = y_full.copy(); y_nan[cmask(y_nan.index)] = np.nan
y_drop = y_full[~cmask(y_full.index)]

# Report any interior NaNs patched in the forecast-period exog (gap-crossing lags).
_patched = df.loc[fc_idx, EXOG].isna().sum()
_patched = _patched[_patched > 0]

print(f'Exog-complete fit window : {start.date()} -> {y_full.index.max().date()}')
print(f'  observations (n)       : {len(y_full)}')
print(f'  COVID quarters -> NaN  : {int(cmask(y_full.index).sum())}  '
      f'({COVID_START} .. {COVID_END})')
print(f'  usable after COVID drop: {len(y_drop)}')
print(f'Forecast horizon         : {fc_idx.min().date()} -> {fc_idx.max().date()}  '
      f'({len(fc_idx)} quarters)')
if len(_patched):
    print('  forecast-exog NaNs patched by interpolation:')
    for k, v in _patched.items(): print(f'    {k}: {int(v)}')
else:
    print('  forecast-exog: no NaNs')

# ── Collinearity audit on the exogenous block (VIF + condition number) ─────────
Xv = X_hist.loc[y_drop.index].dropna()
Xc = np.column_stack([np.ones(len(Xv)), Xv.values])
vif = pd.Series({c: variance_inflation_factor(Xc, i + 1) for i, c in enumerate(Xv.columns)})
cond = np.linalg.cond(Xv.values - Xv.values.mean(0))
print('\nExogenous-block collinearity (VIF; rule-of-thumb concern > 5):')
for c, v in vif.sort_values(ascending=False).items():
    print(f'  {c:<28s} VIF={v:5.2f}   corr(target)={y_drop.reindex(Xv.index).corr(Xv[c]):+.2f}')
print(f'  condition number (centred) = {cond:.1f}')

Exog-complete fit window : 1991-12-31 -> 2025-12-31
  observations (n)       : 137
  COVID quarters -> NaN  : 8  (2020-03-31 .. 2021-12-31)
  usable after COVID drop: 129
Forecast horizon         : 2026-03-31 -> 2030-12-31  (20 quarters)
  forecast-exog NaNs patched by interpolation:
    us_credit_qoq_growth_L6: 1

Exogenous-block collinearity (VIF; rule-of-thumb concern > 5):
  us_unemployment_L1           VIF= 3.05   corr(target)=+0.33
  us_house_price_yoy_L3        VIF= 2.38   corr(target)=-0.50
  us_credit_qoq_growth_L6      VIF= 1.68   corr(target)=+0.35
  us_consumer_confidence_L2    VIF= 1.53   corr(target)=-0.34
  us_gdp_yoy_growth_L0         VIF= 1.49   corr(target)=-0.22
  us_cpi_L6                    VIF= 1.29   corr(target)=+0.19
  condition number (centred) = 7.8


## **1: Sample Diagnostics & EDA — the COVID transmission break**
___
The single most important modelling fact for this target: **the pandemic severed the
macro→credit link.** In every prior recession, rising unemployment pulled delinquency
up (the GFC took it to ~6.8%). During COVID, mass forbearance and stimulus pushed
delinquency *down* to ~1.5% even as unemployment spiked to 11%. A macro-conditional
model fit through that window would learn a relationship that never held structurally —
so those quarters are excluded from fitting and from evaluation.

In [3]:
# Figure 1 — delinquency vs unemployment, with the COVID inversion highlighted
unemp = df['us_unemployment'] if 'us_unemployment' in df.columns else None
fig = make_subplots(specs=[[{'secondary_y': True}]])

for s, e, lbl in RECESSIONS_US:
    fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.3, layer='below', line_width=0)
fig.add_vrect(x0=COVID_START, x1=COVID_END, fillcolor=AMBER, opacity=0.12,
              layer='below', line_width=0,
              annotation_text='COVID excluded', annotation_position='top left',
              annotation_font_size=10)

fig.add_trace(go.Scatter(x=y_full.index, y=y_full.values, mode='lines',
                         line=dict(color=NAVY, width=1.8), name='Delinquency rate (%)'),
              secondary_y=False)
if unemp is not None:
    um = unemp.loc[start:hist_idx.max()]
    fig.add_trace(go.Scatter(x=um.index, y=um.values, mode='lines',
                             line=dict(color=RED, width=1.4, dash='dot'), name='Unemployment (%)'),
                  secondary_y=True)

fig.update_layout(
    title=dict(text='Figure 1 — Delinquency vs unemployment: the COVID inversion<br>'
                    '<sup>Grey = NBER recessions | amber = COVID window excluded from fit & '
                    'evaluation | note the divergence in 2020–21</sup>',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=460,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=95, b=40, l=55, r=55))
fig.update_yaxes(title_text='Delinquency (%)', secondary_y=False, color=NAVY)
fig.update_yaxes(title_text='Unemployment (%)', secondary_y=True, color=RED)
fig.show()

## **2: Model Specifications**
___
Orders are selected **once** (via `auto_arima` on the COVID-dropped target) and then
only re-estimated during the backtest — the same specification-once convention as 02a,
noted as a mild source of optimism in the caveats. The DM/Clark-West helpers are the
**same functions used in 02a** (Harvey-Leybourne-Newbold small-sample correction; the
Clark-West nested-model adjustment), reproduced here so this notebook is self-contained.

- **AR(1)** — requested baseline; a first-order autoregression on the level.
- **SARIMA** — best univariate ARIMA (order chosen by AIC on the dropped series).
- **SARIMAX** — SARIMA + the six exogenous regressors (consumes 02a's forecasts).
- **ARDL** — autoregressive distributed-lag in the exogenous block; hand-rolled with a
  calendar-correct lag matrix so it is unambiguous and auditable. AR order by AIC.
- **VAR** — joint system on the contemporaneous block; differenced if ADF cannot reject
  a unit root; lag by AIC. Independent cross-check (does not consume 02a).

In [4]:
# ── Model-class label (verbatim from 02a) ─────────────────────────────────────
def classify_model(order, seasonal_order):
    p, d, q = order; P, D, Q, m = seasonal_order
    if p == 0 and q == 0 and P == 0 and Q == 0: return 'Mean'
    if P > 0 or Q > 0 or D > 0:                  return 'SARMA'
    if q == 0 and Q == 0:                        return 'AR'
    return 'ARMA'

# ── Diebold-Mariano with HLN small-sample correction (HAC when h>1) — from 02a ──
def dm_hln(e_bench, e_model, h=BACKTEST_H):
    d = e_bench ** 2 - e_model ** 2; n = len(d)      # >0 => model beats benchmark
    if n < 3: return np.nan, np.nan
    var = np.var(d, ddof=0)
    for k in range(1, h):
        var += 2 * np.cov(d[:-k], d[k:])[0, 1]
    var /= n
    if var <= 0: return np.nan, np.nan               # negative LRV -> DM undefined
    dm   = d.mean() / np.sqrt(var)
    corr = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # Harvey-Leybourne-Newbold 1997
    dm_s = dm * corr
    return dm_s, 2 * (1 - stats.t.cdf(abs(dm_s), df=n - 1))

# ── Clark-West adjustment (correct test when the benchmark is NESTED) — from 02a ─
def clark_west(e_bench, e_model, f_bench, f_model):
    ft = e_bench ** 2 - (e_model ** 2 - (f_bench - f_model) ** 2); n = len(ft)
    if n < 3: return np.nan, np.nan
    t = ft.mean() / (ft.std(ddof=1) / np.sqrt(n))
    return t, 1 - stats.norm.cdf(t)                  # one-sided: model better

# ── Hand-rolled ARDL: calendar-correct lag matrix, OLS, recursive forecast ─────
def ardl_design(y, X, p):
    d = pd.DataFrame({'y': y})
    for i in range(1, p + 1): d[f'yL{i}'] = y.shift(i)
    for c in X.columns:       d[c] = X[c]
    d = d.dropna()
    M_ = np.column_stack([np.ones(len(d))]
                         + [d[f'yL{i}'].values for i in range(1, p + 1)]
                         + [d[c].values for c in X.columns])
    return d.index, d['y'].values, M_

def ardl_fit(y, X, p):
    _, Y, Mx = ardl_design(y, X, p)
    b, *_ = np.linalg.lstsq(Mx, Y, rcond=None)
    return b, Y - Mx @ b

def ardl_pick_p(y, X, pmax=4):
    best = (np.inf, 1)
    for p in range(1, pmax + 1):
        _, Y, Mx = ardl_design(y, X, p)
        b, *_ = np.linalg.lstsq(Mx, Y, rcond=None)
        r = Y - Mx @ b; n = len(Y); k = Mx.shape[1]
        aic = n * np.log(np.sum(r ** 2) / n) + 2 * k
        if aic < best[0]: best = (aic, p)
    return best[1]

def ardl_forecast(b, y_obs, Xf, p):
    buf = list(y_obs.dropna().values); cols = list(Xf.columns); out = []
    for h in range(len(Xf)):
        yl = [buf[-i] for i in range(1, p + 1)]
        yhat = b[0] + sum(b[i] * yl[i - 1] for i in range(1, p + 1)) + b[1 + p:] @ Xf.iloc[h][cols].values
        out.append(yhat); buf.append(yhat)
    return np.array(out)

# ── Select orders ONCE ────────────────────────────────────────────────────────
sar = auto_arima(y_drop, seasonal=True, m=M, information_criterion='aic', stepwise=True,
                 error_action='ignore', suppress_warnings=True,
                 max_p=3, max_q=3, max_P=2, max_Q=2)
Xd = X_hist.loc[y_drop.index]
sarx = auto_arima(y_drop, exogenous=Xd.values, seasonal=True, m=M, information_criterion='aic',
                  stepwise=True, error_action='ignore', suppress_warnings=True,
                  max_p=3, max_q=3, max_P=1, max_Q=1)
SARIMA_ORD,  SARIMA_SORD  = sar.order,  sar.seasonal_order
SARIMAX_ORD, SARIMAX_SORD = sarx.order, sarx.seasonal_order
ARDL_P = ardl_pick_p(y_nan, X_hist)

# ── VAR differencing decision + lag (fixed ONCE for reuse) ─────────────────────
Vfull = df.loc[start:hist_idx.max(), VAR_VARS]
V0 = Vfull.dropna(); V0 = V0[~cmask(V0.index)]
VAR_DIFF = any((adfuller(V0[c].dropna(), autolag='AIC')[1] >= 0.05) for c in V0.columns)
Vm0 = V0.diff().dropna() if VAR_DIFF else V0
VAR_LAG = VAR(Vm0).fit(maxlags=4, ic='aic').k_ar

print('Selected specifications')
print(f'  AR(1)   : (1,0,0)')
print(f'  SARIMA  : {SARIMA_ORD} x {SARIMA_SORD}  [{classify_model(SARIMA_ORD, SARIMA_SORD)}]')
print(f'  SARIMAX : {SARIMAX_ORD} x {SARIMAX_SORD} + {len(EXOG)} exog')
print(f'  ARDL    : p={ARDL_P}  + {len(EXOG)} exog')
print(f'  VAR     : {len(VAR_VARS)} vars | differenced={VAR_DIFF} | lag={VAR_LAG}')

Selected specifications
  AR(1)   : (1,0,0)
  SARIMA  : (1, 1, 0) x (0, 0, 0, 4)  [AR]
  SARIMAX : (1, 1, 0) x (0, 0, 0, 4) + 6 exog
  ARDL    : p=1  + 6 exog
  VAR     : 5 vars | differenced=True | lag=4


## **3: Out-of-Sample Backtest & Diebold-Mariano**
___
RMSE and the Diebold-Mariano test are *out-of-sample* quantities, so we run a
**recursive (expanding-window) rolling-origin backtest**: at each origin the selected
spec is re-estimated on data up to that point, a one-step forecast is made against the
naive random walk, and the error is recorded (Diebold & Mariano 1995).

**Design (mirrors 02a).**
- *Scheme:* expanding window — maximises training data at this sample size.
- *Horizon:* `h = 1`. One-step loss differentials are serially uncorrelated (cleanest DM);
  crucially, at `h=1` every lagged regressor is already realised, so the conditional models
  are tested fairly with **no** forecasted inputs.
- *Benchmark:* naive random walk.
- *`beats_RW` flag:* a random walk is nested in the differenced ARIMA specs, where the
  standard DM test is invalid — so the flag uses the **Clark-West** nested-model test and
  additionally requires the model to beat naive in realised RMSE, so it never contradicts
  the RMSE column. The HLN-DM statistic is reported alongside for transparency.
- Origins whose target lands in the COVID window are skipped.

A focused **HLN-DM** then compares the best macro-conditional model against the best
pure-time-series model (a non-nested, two-sided test) to answer directly: *does adding
the macro block beat a good univariate model out-of-sample?*

In [5]:
# ── One-step forecast from a training window (NaN-in-place already applied) ────
def refit_forecast(name, tr_y, tr_X, fut_X):
    if name == 'Naive RW':
        return float(tr_y.dropna().iloc[-1])
    if name == 'AR(1)':
        m = SARIMAX(tr_y, order=(1, 0, 0), enforce_stationarity=False,
                    enforce_invertibility=False).fit(disp=False, maxiter=100)
        return float(m.forecast(1).iloc[-1])
    if name == 'SARIMA':
        m = SARIMAX(tr_y, order=SARIMA_ORD, seasonal_order=SARIMA_SORD,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
        return float(m.forecast(1).iloc[-1])
    if name == 'SARIMAX':
        m = SARIMAX(tr_y, exog=tr_X, order=SARIMAX_ORD, seasonal_order=SARIMAX_SORD,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
        return float(m.get_forecast(1, exog=fut_X).predicted_mean.iloc[-1])
    if name == 'ARDL':
        bb, _ = ardl_fit(tr_y, tr_X, ARDL_P)
        return float(ardl_forecast(bb, tr_y, fut_X, ARDL_P)[0])
    if name == 'VAR':
        Vt = tr_X                                   # tr_X carries the VAR block here
        Vmm = Vt.diff().dropna() if VAR_DIFF else Vt
        r = VAR(Vmm).fit(maxlags=4, ic='aic')
        f = pd.DataFrame(r.forecast(Vmm.values[-r.k_ar:], steps=1), columns=Vmm.columns)[TARGET].values[0]
        return float(Vt[TARGET].iloc[-1] + f) if VAR_DIFF else float(f)

# ── Recursive rolling-origin loop ─────────────────────────────────────────────
MODELS = ['Naive RW', 'AR(1)', 'SARIMA', 'SARIMAX', 'ARDL', 'VAR']
qi = y_nan.index
err = {m: [] for m in MODELS}; fcv = {m: [] for m in MODELS}; act = []; dts = []
n_used = 0
for t in range(BACKTEST_MIN_TRAIN, len(y_nan) - BACKTEST_H + 1):
    tgt = t + BACKTEST_H - 1
    if (qi[tgt].to_period('Q') - qi[t - 1].to_period('Q')).n != BACKTEST_H:   # gap guard
        continue
    if cmask([qi[tgt]])[0]:                                                    # target in COVID
        continue
    a = y_nan.iloc[tgt]
    if not np.isfinite(a):
        continue
    tr_y = y_nan.iloc[:t]; tr_X = X_hist.iloc[:t]; fut_X = X_hist.iloc[[tgt]]
    tr_V = Vfull.iloc[:t].dropna()
    try:
        preds = {}
        for m in MODELS:
            if   m == 'VAR':                     preds[m] = refit_forecast(m, tr_y, tr_V, None)
            elif m in ('SARIMAX', 'ARDL'):       preds[m] = refit_forecast(m, tr_y, tr_X, fut_X)
            else:                                preds[m] = refit_forecast(m, tr_y, None, None)
    except Exception:
        continue
    if not all(np.isfinite(list(preds.values()))):
        continue
    for m in MODELS:
        err[m].append(a - preds[m]); fcv[m].append(preds[m])
    act.append(a); dts.append(qi[tgt]); n_used += 1
print(f'Recursive expanding-window backtest | h={BACKTEST_H} | benchmark = naive RW')
print(f'Origins used: {n_used}  ({pd.Timestamp(dts[0]).date()} -> {pd.Timestamp(dts[-1]).date()})\n')

# ── Error metrics (MSE / RMSE / MAE / MAPE) + skill vs naive RW ────────────────
# Delinquency is strictly positive (min ~1.5%), so MAPE is well-defined here.
a_arr = np.array(act)
en = np.array(err['Naive RW']); fn = np.array(fcv['Naive RW'])
rmse_n = np.sqrt(np.mean(en ** 2))

res_rows, diag_rows = [], []
for m in MODELS:
    em = np.array(err[m]); fm = np.array(fcv[m])
    mse  = float(np.mean(em ** 2))
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(em)))
    mape = float(np.mean(np.abs(em / a_arr)) * 100)
    if m == 'Naive RW':
        res_rows.append({'model': m, 'MSE': round(mse, 4), 'RMSE': round(rmse, 4),
                         'MAE': round(mae, 4), 'MAPE%': round(mape, 2),
                         'skill%': 0.0, 'beats_RW': '—'})
        diag_rows.append({'model': m, 'DM*': None, 'DM_p': None, 'CW': None, 'CW_p': None})
        continue
    skill = 100 * (1 - rmse / rmse_n)
    dm, pdm = dm_hln(en, em, BACKTEST_H); cw, pcw = clark_west(en, em, fn, fm)
    beats = (rmse < rmse_n) and np.isfinite(pcw) and pcw < 0.05
    res_rows.append({'model': m, 'MSE': round(mse, 4), 'RMSE': round(rmse, 4),
                     'MAE': round(mae, 4), 'MAPE%': round(mape, 2),
                     'skill%': round(skill, 1), 'beats_RW': 'Yes' if beats else 'No'})
    diag_rows.append({'model': m,
                      'DM*': round(dm, 3) if np.isfinite(dm) else None,
                      'DM_p': round(pdm, 4) if np.isfinite(pdm) else None,
                      'CW': round(cw, 3) if np.isfinite(cw) else None,
                      'CW_p': round(pcw, 4) if np.isfinite(pcw) else None})

results = pd.DataFrame(res_rows).set_index('model').sort_values('RMSE')
diag    = pd.DataFrame(diag_rows).set_index('model').loc[results.index]

# ── Results table (headline) — error metrics + skill, best first ───────────────
print('Results (MSE/RMSE/MAE in native % units; MAPE scale-free; skill vs naive RW):')
display(results)

# ── Diagnostics table (significance tests) — mirrors 02a's split ───────────────
print('Diagnostics — DM* = HLN-corrected Diebold-Mariano (Harvey-Leybourne-Newbold), '
      'CW = Clark-West nested-benchmark test (one-sided); both >0 favour the model, '
      'p < 0.05 = significant:')
display(diag)

# ── Focused non-nested test: best macro-conditional vs best pure-TS ────────────
ts_pool    = {m: np.sqrt(np.mean(np.array(err[m]) ** 2)) for m in ['AR(1)', 'SARIMA']}
macro_pool = {m: np.sqrt(np.mean(np.array(err[m]) ** 2)) for m in ['SARIMAX', 'ARDL']}
best_ts    = min(ts_pool,    key=ts_pool.get)
best_macro = min(macro_pool, key=macro_pool.get)
dm_h, p_h  = dm_hln(np.array(err[best_ts]), np.array(err[best_macro]), BACKTEST_H)
print(f'Does macro beat pure time-series?  {best_macro} vs {best_ts} '
      f'(HLN-DM, two-sided, non-nested):')
print(f'  DM* = {dm_h:+.3f} | p = {p_h:.4f}  ->  '
      f'{"macro significantly better" if (p_h < 0.05 and dm_h > 0) else "no significant difference at 5%"}')

# ── booktabs LaTeX export of the results table (mirrors 02a) ───────────────────
def _tex_escape(s): return str(s).replace('_', r'\_')
def report_to_latex(df):
    out = [r'\begin{tabular}{lrrrrrc}', r'  \toprule',
           r'  Model & MSE & RMSE & MAE & MAPE\% & Skill\% & Beats RW \\', r'  \midrule']
    for m, r in df.iterrows():
        out.append('  {} & {} & {} & {} & {} & {} & {} \\\\'.format(
            _tex_escape(m), r['MSE'], r['RMSE'], r['MAE'], r['MAPE%'],
            '--' if pd.isna(r['skill%']) else r['skill%'], r['beats_RW']))
    out += [r'  \bottomrule', r'\end{tabular}']
    return '\n'.join(out)

latex_table = report_to_latex(results)
try:
    (LOCAL_OUTPUT / f'{MODEL_PREFIX}_delinquency_backtest_US.tex').write_text(latex_table)
    print(f'LaTeX results table -> {LOCAL_OUTPUT / (MODEL_PREFIX + "_delinquency_backtest_US.tex")}')
except Exception as e:
    print(f'(LaTeX not written to disk: {e})')
print(latex_table)

Recursive expanding-window backtest | h=1 | benchmark = naive RW
Origins used: 69  (2006-12-31 -> 2025-12-31)

Results (MSE/RMSE/MAE in native % units; MAPE scale-free; skill vs naive RW):


,MSE,RMSE,MAE,MAPE%,skill%,beats_RW
model,,,,,,
ARDL,0.0369,0.1920,0.1348,4.39,26.3,Yes
SARIMAX,0.0454,0.2132,0.1366,3.77,18.1,Yes
SARIMA,0.0470,0.2168,0.1149,3.42,16.7,Yes
VAR,0.0538,0.2319,0.1417,4.19,11.0,Yes
AR(1),0.0646,0.2542,0.1618,4.73,2.4,No
Naive RW,0.0678,0.2604,0.1626,4.81,0.0,—


Diagnostics — DM* = HLN-corrected Diebold-Mariano (Harvey-Leybourne-Newbold), CW = Clark-West nested-benchmark test (one-sided); both >0 favour the model, p < 0.05 = significant:


,DM*,DM_p,CW,CW_p
model,,,,
ARDL,2.323,0.0232,3.546,0.0002
SARIMAX,1.634,0.1069,2.621,0.0044
SARIMA,3.088,0.0029,3.483,0.0002
VAR,0.794,0.4299,1.962,0.0249
AR(1),0.778,0.4394,0.881,0.1891
Naive RW,NaN,NaN,NaN,NaN


Does macro beat pure time-series?  ARDL vs SARIMA (HLN-DM, two-sided, non-nested):
  DM* = +1.106 | p = 0.2728  ->  no significant difference at 5%
LaTeX results table -> /Users/dannyhogan/Desktop/ST-498/Data Collection/TS_delinquency_backtest_US.tex
\begin{tabular}{lrrrrrc}
  \toprule
  Model & MSE & RMSE & MAE & MAPE\% & Skill\% & Beats RW \\
  \midrule
  ARDL & 0.0369 & 0.192 & 0.1348 & 4.39 & 26.3 & Yes \\
  SARIMAX & 0.0454 & 0.2132 & 0.1366 & 3.77 & 18.1 & Yes \\
  SARIMA & 0.047 & 0.2168 & 0.1149 & 3.42 & 16.7 & Yes \\
  VAR & 0.0538 & 0.2319 & 0.1417 & 4.19 & 11.0 & Yes \\
  AR(1) & 0.0646 & 0.2542 & 0.1618 & 4.73 & 2.4 & No \\
  Naive RW & 0.0678 & 0.2604 & 0.1626 & 4.81 & 0.0 & — \\
  \bottomrule
\end{tabular}


In [6]:
# Figure 2 — one-step OOS predictions vs actual (all models)
oos = pd.DataFrame({'actual': act}, index=pd.DatetimeIndex(dts))
for m in MODELS: oos[m] = fcv[m]
colmap = {'Naive RW': GREY, 'AR(1)': AMBER, 'SARIMA': GREEN,
          'SARIMAX': BLUE, 'ARDL': RED, 'VAR': '#8E44AD'}

fig = go.Figure()
fig.add_trace(go.Scatter(x=oos.index, y=oos['actual'], mode='lines',
                         line=dict(color=NAVY, width=2.4), name='Actual'))
for m in MODELS:
    fig.add_trace(go.Scatter(x=oos.index, y=oos[m], mode='lines',
                             line=dict(color=colmap[m], width=1.2,
                                       dash='dot' if m == 'Naive RW' else 'solid'),
                             opacity=0.9, name=m))
fig.update_layout(
    title=dict(text='Figure 2 — One-step out-of-sample predictions vs actual<br>'
                    '<sup>Expanding-window recursive backtest | COVID targets excluded</sup>',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=460, yaxis_title='Delinquency rate (%)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=95, b=40, l=55, r=20))
fig.show()

## **4: 20-Quarter Forward Forecast (2026–2030)**
___
Each model is re-fit on the full NaN-in-place history and projected 20 quarters. The
conditional models (SARIMAX, ARDL) are driven by **02a's forecasted regressors**; the
VAR generates its own paths. A 95% band is shown for the SARIMAX conditional mean only
— and, per the note in Cell 0, that band still treats the upstream macro forecasts as
known, so it is a *lower bound* on true forecast uncertainty. Full uncertainty
propagation and IFRS 9 scenario construction belong to a later stage.

Note the regressors phase in as their lags reach the horizon: only `us_gdp_yoy_growth_L0`
is forecast-dependent from 2026Q1; unemployment (L1) from 2026Q2, confidence (L2) from
2026Q3, house prices (L3) from 2026Q4, and credit/CPI (L6) from 2027Q3.

In [7]:
# ── SARIMAX/AR/SARIMA forward via state space (NaN-in-place history) ───────────
def sx_forward(order, sord, exog_hist, exog_fc):
    m = SARIMAX(y_nan, exog=exog_hist, order=order, seasonal_order=sord,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
    fo = m.get_forecast(H, exog=exog_fc); ci = fo.conf_int(alpha=0.05)
    return fo.predicted_mean.values, ci.iloc[:, 0].values, ci.iloc[:, 1].values

fwd = {}; ci_lo = {}; ci_hi = {}
fwd['Naive RW'] = np.repeat(y_full.dropna().iloc[-1], H)
fwd['AR(1)'],   ci_lo['AR(1)'],   ci_hi['AR(1)']   = sx_forward((1, 0, 0), (0, 0, 0, 0), None, None)
fwd['SARIMA'],  ci_lo['SARIMA'],  ci_hi['SARIMA']  = sx_forward(SARIMA_ORD, SARIMA_SORD, None, None)
fwd['SARIMAX'], ci_lo['SARIMAX'], ci_hi['SARIMAX'] = sx_forward(SARIMAX_ORD, SARIMAX_SORD, X_hist, X_fc)

# ARDL forward (recursive, hand-rolled)
b, _ = ardl_fit(y_nan, X_hist, ARDL_P)
fwd['ARDL'] = ardl_forecast(b, y_nan, X_fc, ARDL_P)

# VAR forward (joint system; reconstruct level if differenced)
Vm = V0.diff().dropna() if VAR_DIFF else V0
vres = VAR(Vm).fit(maxlags=4, ic='aic')
vf = pd.DataFrame(vres.forecast(Vm.values[-vres.k_ar:], steps=H), columns=Vm.columns)[TARGET].values
if VAR_DIFF: vf = V0[TARGET].iloc[-1] + np.cumsum(vf)
fwd['VAR'] = vf

forward = pd.DataFrame(fwd, index=fc_idx)
last_obs = float(y_full.dropna().iloc[-1])
print(f'Last observed delinquency ({y_full.dropna().index[-1].date()}): {last_obs:.3f}%\n')
print('20-quarter forward summary (delinquency %):')
print(f'  {"model":<10s} {"2026Q1":>8s} {"2030Q4":>8s} {"min":>7s} {"mean":>7s} {"max":>7s}')
for m in MODELS:
    v = forward[m].values
    print(f'  {m:<10s} {v[0]:8.3f} {v[-1]:8.3f} {v.min():7.3f} {v.mean():7.3f} {v.max():7.3f}')
print(f'\nSpread at 2030Q4: ARDL {forward["ARDL"].iloc[-1]:.2f}%  (highest)  vs  '
      f'VAR {forward["VAR"].iloc[-1]:.2f}%  (lowest)  — a ~{forward["ARDL"].iloc[-1] - forward["VAR"].iloc[-1]:.1f} pp gap.')
print('This ARDL-vs-VAR divergence is the model-choice risk to flag for scenario work: '
      'the exog-driven ARDL extrapolates the recent rise, while the joint VAR mean-reverts.')

Last observed delinquency (2025-12-31): 2.940%

20-quarter forward summary (delinquency %):
  model        2026Q1   2030Q4     min    mean     max
  Naive RW      2.940    2.940   2.940   2.940   2.940
  AR(1)         2.921    2.587   2.587   2.751   2.921
  SARIMA        2.918    2.892   2.892   2.895   2.918
  SARIMAX       2.925    2.968   2.901   2.957   2.979
  ARDL          2.994    3.934   2.933   3.523   3.934
  VAR           2.873    2.361   2.361   2.506   2.873

Spread at 2030Q4: ARDL 3.93%  (highest)  vs  VAR 2.36%  (lowest)  — a ~1.6 pp gap.
This ARDL-vs-VAR divergence is the model-choice risk to flag for scenario work: the exog-driven ARDL extrapolates the recent rise, while the joint VAR mean-reverts.


In [8]:
# Figure 3 — forward forecasts (last 10y history + 20q horizon), SARIMAX band shown
hist_tail = y_full.dropna()
cutoff = hist_tail.index[-1] - pd.DateOffset(years=10)
hist_tail = hist_tail[hist_tail.index >= cutoff]

fig = go.Figure()
fig.add_vrect(x0=COVID_START, x1=COVID_END, fillcolor=AMBER, opacity=0.10, layer='below', line_width=0)
fig.add_trace(go.Scatter(x=hist_tail.index, y=hist_tail.values, mode='lines',
                         line=dict(color=NAVY, width=2.0), name='Historical'))

# SARIMAX 95% band (conditional mean only; see caveat)
fig.add_trace(go.Scatter(x=list(fc_idx) + list(fc_idx[::-1]),
                         y=list(ci_hi['SARIMAX']) + list(ci_lo['SARIMAX'][::-1]),
                         fill='toself', fillcolor='rgba(46,117,182,0.15)',
                         line=dict(width=0), hoverinfo='skip', showlegend=True,
                         name='SARIMAX 95% (macro known)'))
for m in MODELS:
    fig.add_trace(go.Scatter(x=fc_idx, y=forward[m].values, mode='lines',
                             line=dict(color=colmap[m], width=1.8,
                                       dash='dot' if m == 'Naive RW' else 'solid'),
                             name=m))
fig.update_layout(
    title=dict(text='Figure 3 — Delinquency: 20-quarter forward forecast (2026–2030)<br>'
                    '<sup>Last 10y history | shaded band = SARIMAX 95% treating 02a macro '
                    'forecasts as known (lower bound on true uncertainty)</sup>',
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=480, yaxis_title='Delinquency rate (%)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=100, b=40, l=55, r=20))
fig.show()

## **5: Export & Limitations**
___
The forward paths are written to `TS_delinquency_forecast_US_Q.csv` for comparison
against the ML PD model. This is the time-series **benchmark** deliverable — not a
replacement for the ML model, and not the final ECL scenario set.

In [9]:
# ── Save forward forecasts + SARIMAX band ─────────────────────────────────────
out = forward.copy()
out['SARIMAX_lo95'] = ci_lo['SARIMAX']
out['SARIMAX_hi95'] = ci_hi['SARIMAX']
out.index.name = 'date'

saved_to = None
for target_dir in [LOCAL_OUTPUT, Path.cwd()]:
    try:
        target_dir.mkdir(parents=True, exist_ok=True)
        out.to_csv(target_dir / OUT_FORECAST); saved_to = target_dir / OUT_FORECAST; break
    except Exception:
        continue
print(f'Saved {OUT_FORECAST}  {out.shape}  ->  {saved_to}')
print(out.round(3).to_string())

print('''
─────────────────────────────────────────────────────────────────────────────
LIMITATIONS (state in the write-up)
─────────────────────────────────────────────────────────────────────────────
1. Exogenous uncertainty not propagated. SARIMAX/ARDL treat 02a's forecasted
   regressors as known; the plotted band is therefore a LOWER BOUND on true
   forecast uncertainty. Propagation + IFRS 9 base/upside/downside are a later stage.
2. One-step backtest != five-year path. h=1 uses realised lagged regressors, so it
   validates short-horizon conditional skill, not the full lifetime projection.
3. COVID excluded (fit + eval). The 2020–21 forbearance break inverted the macro→
   credit link; those quarters are NaN and skipped, not modelled.
4. VAR is an independent cross-check, not a consumer of Stage 1. Its divergence from
   ARDL bounds model-choice risk for the scenario stage.
5. Specification chosen once. Orders are selected on the full dropped sample and only
   re-estimated per origin — mildly optimistic RMSE (re-run auto_arima in-loop to remove).
6. US proof-of-concept. The UK pipeline requires re-sourcing the UK target and re-running.
''')

Saved TS_delinquency_forecast_US_Q.csv  (20, 8)  ->  /Users/dannyhogan/Desktop/ST-498/Data Collection/TS_delinquency_forecast_US_Q.csv
            Naive RW  AR(1)  SARIMA  SARIMAX   ARDL    VAR  SARIMAX_lo95  SARIMAX_hi95
date                                                                                  
2026-03-31      2.94  2.921   2.918    2.925  2.994  2.873         2.597         3.253
2026-06-30      2.94  2.903   2.906    2.901  2.933  2.754         2.354         3.447
2026-09-30      2.94  2.884   2.900    2.931  2.994  2.649         2.208         3.654
2026-12-31      2.94  2.866   2.896    2.938  3.068  2.582         2.067         3.809
2027-03-31      2.94  2.847   2.894    2.946  3.167  2.589         1.946         3.945
2027-06-30      2.94  2.829   2.893    2.954  3.291  2.564         1.841         4.067
2027-09-30      2.94  2.811   2.893    2.979  3.449  2.543         1.762         4.195
2027-12-31      2.94  2.793   2.892    2.971  3.514  2.512         1.659         4